# Classificadores Multiclasse Baseados em Algoritmos de Agrupamento

Implementação em Python para os datasets Adult e Dry Bean.

In [ ]:
# Importação de bibliotecas
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Função para encontrar o número ideal de clusters (Elbow Method)

In [ ]:
def elbow_method(X, max_k=10):
    distortions = []
    for k in range(1, max_k+1):
        kmeans = KMeans(n_clusters=k, random_state=0)
        kmeans.fit(X)
        distortions.append(kmeans.inertia_)
    plt.plot(range(1, max_k+1), distortions, marker='o')
    plt.xlabel('Número de clusters')
    plt.ylabel('Distortion')
    plt.title('Método do Cotovelo')
    plt.show()
    # Retorne o valor de k sugerido visualmente
    return np.argmin(np.diff(distortions, 2)) + 2

## Função para treinar e avaliar classificadores baseados em agrupamento

In [ ]:
def cluster_classifier(X_train, y_train, X_test, y_test, clusterer):
    # Ajusta o clusterer nos dados de treino
    clusterer.fit(X_train)
    # Associa cada cluster à classe majoritária
    clusters = clusterer.labels_ if hasattr(clusterer, 'labels_') else clusterer.predict(X_train)
    cluster_to_class = {}
    for c in np.unique(clusters):
        mask = clusters == c
        majority_class = pd.Series(y_train[mask]).mode()[0]
        cluster_to_class[c] = majority_class
    # Predição nos dados de teste
    test_clusters = clusterer.predict(X_test)
    y_pred = np.array([cluster_to_class.get(c, -1) for c in test_clusters])
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    return acc, cm

## Função para rodar experimentos repetidos

In [ ]:
def run_experiments(X, y, clusterer_class, n_clusters, n_runs=30):
    accs = []
    cms = []
    for seed in range(1, n_runs+1):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y)
        clusterer = clusterer_class(n_clusters=n_clusters, random_state=seed)
        acc, cm = cluster_classifier(X_train, y_train, X_test, y_test, clusterer)
        accs.append(acc)
        cms.append(cm)
    return np.array(accs), cms

## Carregamento e preparação dos dados (Exemplo: Adult)
Repita para Dry Bean conforme necessário.

In [ ]:
# Substitua pelo caminho correto do arquivo Adult
adult = pd.read_csv('adult.csv')
# Pré-processamento
adult = adult.dropna()
for col in adult.select_dtypes(include='object').columns:
    adult[col] = LabelEncoder().fit_transform(adult[col])
X_adult = adult.drop('class', axis=1).values
y_adult = adult['class'].values
X_adult = StandardScaler().fit_transform(X_adult)

## Encontrar número ideal de clusters para Adult

In [ ]:
k_adult = elbow_method(X_adult, max_k=10)
print(f'Número ideal de clusters (Adult): {k_adult}')

## Executar experimentos com três classificadores de agrupamento

In [ ]:
# KMeans
accs_kmeans, cms_kmeans = run_experiments(X_adult, y_adult, KMeans, k_adult)
# Agglomerative
accs_agg, cms_agg = run_experiments(X_adult, y_adult, AgglomerativeClustering, k_adult)
# Spectral
accs_spec, cms_spec = run_experiments(X_adult, y_adult, SpectralClustering, k_adult)
print('KMeans:', accs_kmeans.mean(), accs_kmeans.std())
print('Agglomerative:', accs_agg.mean(), accs_agg.std())
print('Spectral:', accs_spec.mean(), accs_spec.std())

## Exibir matriz de confusão média

In [ ]:
def plot_mean_cm(cms, labels):
    mean_cm = np.mean(cms, axis=0)
    plt.figure(figsize=(6,5))
    sns.heatmap(mean_cm, annot=True, fmt='.0f', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predito')
    plt.ylabel('Verdadeiro')
    plt.title('Matriz de Confusão Média')
    plt.show()

# Exemplo para KMeans
plot_mean_cm(cms_kmeans, np.unique(y_adult))

## Repita o mesmo processo para o dataset Dry Bean
### (Carregue, pré-processe, encontre k, execute experimentos e avalie)